In [2]:
# install
!pip install pesq
!pip install pystoi
!pip install tensorboardX

  Preparing metadata (setup.py) ... done
  Created wheel for pesq: filename=pesq-0.0.4-cp310-cp310-linux_x86_64.whl size=262959 sha256=27184c4ff7187cc7d67a8d659527677caac83c0796ccfce3bc477ccbcc3049cb
  Stored in directory: /root/.cache/pip/wheels/c5/4e/2c/251524370c0fdd659e99639a0fbd0ca5a782c3aafcd456b28d
Successfully built pesq
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.6 MB/s eta 0:00:00


In [3]:
# libraries
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import datetime
import random
import numpy as np
import time
import soundfile
from easydict import EasyDict
from pesq import pesq
from pystoi import stoi
from joblib import Parallel, delayed
import numpy as np
import torch.nn.functional as functional
from scipy.signal import get_window
import matplotlib.pylab as plt
from tensorboardX import SummaryWriter
import librosa
import io
import PIL

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [50]:
# options
args = EasyDict()
args.batch_size = 10
args.nepoch = 100
args.optimizer = 'adamW'
args.lr_initial = 5e-4
args.decay_epoch = 30
args.weight_decay = 0.02

args.arch = 'ED_FNN'
args.disc = False
args.loss_type = 'base'
args.loss_oper = 'l2'
args.c = [0.1, 0.9, 0.5, 0.5]
args.device = 'cuda'
args.input_type = 'mag'
args.target_type = 'masking'

args.hidd_ch = [32, 64, 128, 256]
args.norm_layer = 'bn'
args.act = 'relu'
args.kernel_size = (3, 2)
args.stride = (2, 1)
args.dilation = (1, 1)

args.env = 'base'
args.pretrained = False
args.pretrained_init = False
args.pretrain_model_path = None

args.database = 'VBD'
args.fft_len = 512
args.win_len = 400
args.hop_len = 100
args.fs = 16000
args.chunk_size = 32000
args.norm_type = 'None'
args.cmpr = 0.2

args.noisy_dirs_for_train = '/content/gdrive/MyDrive/VBD/train/noisy/'  # YOUR ADDR
args.noisy_dirs_for_valid = '/content/gdrive/MyDrive/VBD/test/noisy/'  # YOUR ADDR

## dataloader

In [6]:
def create_dataloader(opt, mode):
    if mode == 'train':
        return DataLoader(
            dataset=Wave_Dataset(opt, mode),
            batch_size=opt.batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            drop_last=True,
            sampler=None
        )
    elif mode == 'valid':
        return DataLoader(
            dataset=Wave_Dataset(opt, mode),
            batch_size=opt.batch_size, shuffle=False, num_workers=0
        )

In [7]:
class Wave_Dataset(Dataset):
    def __init__(self, opt, mode):
        # load data
        self.mode = mode
        self.chunk_size = opt.chunk_size
        self.norm = opt.norm_type
        self.fs = opt.fs

        if mode == 'train':
            print('<Training dataset>')
            print('Load the data...')
            # load the wav addr
            self.noisy_dirs = scan_directory(opt.noisy_dirs_for_train)
            self.clean_dirs = find_pair(self.noisy_dirs)

        elif mode == 'valid':
            print('<Validation dataset>')
            print('Load the data...')
            # load the wav addr
            self.noisy_dirs = scan_directory(opt.noisy_dirs_for_valid)
            self.clean_dirs = find_pair(self.noisy_dirs)

    def __len__(self):
        return len(self.noisy_dirs)

    def __getitem__(self, idx):
        # read the wav
        inputs, targets = addr2wav(self.noisy_dirs[idx], self.clean_dirs[idx], fs_set=self.fs, norm_type=self.norm)

        # transform to torch from numpy
        inputs = torch.from_numpy(inputs)
        targets = torch.from_numpy(targets)

        wav_len = len(inputs)
        assert wav_len == len(targets)

        if wav_len < self.chunk_size:
            units = self.chunk_size // wav_len
            inputs_final = []
            targets_final = []
            for i in range(units):
                inputs_final.append(inputs)
                targets_final.append(targets)
            inputs_final.append(inputs[:self.chunk_size % wav_len])
            targets_final.append(targets[:self.chunk_size % wav_len])
            inputs = torch.cat(inputs_final, dim=-1)
            targets = torch.cat(targets_final, dim=-1)
        # Randomly crop waveforms to the desired chunk size
        else:
            stp = random.randint(0, len(inputs) - self.chunk_size)
            inputs = inputs[stp:stp + self.chunk_size]
            targets = targets[stp:stp + self.chunk_size]

        return inputs, targets

## models

### baseBlocks

In [8]:
class fcLayer(nn.Module):
    def __init__(self, in_dim, out_dim, norm='bn', act='relu'):
        super(fcLayer, self).__init__()

        self.linear = nn.Linear(in_dim, out_dim)
        self.norm = get_normalization_layer(out_dim, norm=norm)
        self.act = get_activation_layer(act=act)

    def forward(self, x):
        out = self.linear(x).permute(0, 2, 1)
        out = self.norm(out)
        out = self.act(out).permute(0, 2, 1)
        return out


class complexFcLayer(nn.Module):
    def __init__(self, in_dim, out_dim, norm='bn', act='relu'):
        super(complexFcLayer, self).__init__()

        self.linear_r = nn.Linear(in_dim, out_dim)
        self.linear_i = nn.Linear(in_dim, out_dim)

        self.norm_r = get_normalization_layer(out_dim, norm=norm)
        self.act_r = get_activation_layer(act=act)
        self.norm_i = get_normalization_layer(out_dim, norm=norm)
        self.act_i = get_activation_layer(act=act)

    def forward(self, x_r, x_i):
        r2r = self.linear_r(x_r)
        r2i = self.linear_i(x_r)
        i2i = self.linear_i(x_i)
        i2r = self.linear_r(x_i)

        real_out = r2r - i2i
        imag_out = i2r + r2i

        real_out = self.act_r(self.norm_r(real_out.permute(0, 2, 1))).permute(0, 2, 1)
        imag_out = self.act_i(self.norm_i(imag_out.permute(0, 2, 1))).permute(0, 2, 1)

        return real_out, imag_out

In [9]:
def get_padding(kernel_size, dilation):
    return int((kernel_size[0] * dilation[0] - dilation[0]) / 2), int((kernel_size[1] * dilation[1] - dilation[1]))


class causalConv(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size, stride=(1, 1), dilation=(1, 1), groups=1, bias=True):
        super(causalConv, self).__init__()
        padding = get_padding(kernel_size, dilation)
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=kernel_size, stride=stride, padding=(padding[0], 0),
                              dilation=dilation, groups=groups, bias=bias)
        self.padding = padding[1]

    def forward(self, x):
        x = functional.pad(x, [self.padding, 0, 0, 0])
        out = self.conv(x)
        return out


class causalConvTrans(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size, stride=(1, 1), dilation=(1, 1), padding=(0, 0), output_padding=(0, 0)):
        super(causalConvTrans, self).__init__()
        padding = get_padding(kernel_size, dilation)
        self.conv = nn.ConvTranspose2d(in_dim, out_dim, kernel_size, stride=stride, padding=padding,
                                       output_padding=output_padding, dilation=dilation)
        self.padding = padding[1]

    def forward(self, x):
        x = functional.pad(x, [self.padding, 0, 0, 0])
        out = self.conv(x)
        return out


class convLayer(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size=(3, 2), stride=(2, 1), dilation=(1, 1), norm='bn', act='relu'):
        super(convLayer, self).__init__()

        self.conv = causalConv(in_dim, out_dim, kernel_size=kernel_size, stride=stride, dilation=dilation)
        self.norm = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act = get_activation_layer(act=act)

    def forward(self, x):
        out = self.conv(x)
        out = self.norm(out)
        out = self.act(out)
        return out


class complexConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size=(3, 2), stride=(2, 1), dilation=(1, 1), norm='bn', act='relu'):
        super(complexConvLayer, self).__init__()

        self.conv_r = causalConv(in_dim, out_dim, kernel_size=kernel_size, stride=stride, dilation=dilation)
        self.conv_i = causalConv(in_dim, out_dim, kernel_size=kernel_size, stride=stride, dilation=dilation)

        self.norm_r = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act_r = get_activation_layer(act=act)
        self.norm_i = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act_i = get_activation_layer(act=act)

    def forward(self, x_r, x_i):
        r2r = self.conv_r(x_r)
        r2i = self.conv_i(x_r)
        i2i = self.conv_i(x_i)
        i2r = self.conv_r(x_i)

        real_out = r2r - i2i
        imag_out = i2r + r2i

        real_out = self.act_r(self.norm_r(real_out))
        imag_out = self.act_i(self.norm_i(imag_out))

        return real_out, imag_out


class upConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size=(3, 2), stride=(2, 1), dilation=(1, 1), norm='bn', act='relu'):
        super(upConvLayer, self).__init__()

        output_padding = [0, 0]
        if stride[0] > 1:
            output_padding[0] = stride[0] - 1
        else:
            output_padding[0] = 0
        if stride[1] > 1:
            output_padding[1] = stride[1] - 1
        else:
            output_padding[1] = 0

        self.deconv = causalConvTrans(in_dim, out_dim, kernel_size, stride=stride,
                                      padding=get_padding(kernel_size, dilation),
                                      output_padding=output_padding, dilation=dilation)
        self.norm = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act = get_activation_layer(act=act)

    def forward(self, x):
        out = self.deconv(x)
        out = self.norm(out)
        out = self.act(out)
        return out


class complexUpConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim, kernel_size=(3, 2), stride=(2, 1), dilation=(1, 1), norm='bn', act='relu'):
        super(complexUpConvLayer, self).__init__()

        self.conv_r = upConvLayer(in_dim, out_dim, kernel_size=kernel_size, stride=stride, dilation=dilation)
        self.conv_i = upConvLayer(in_dim, out_dim, kernel_size=kernel_size, stride=stride, dilation=dilation)

        self.norm_r = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act_r = get_activation_layer(act=act)
        self.norm_i = get_normalization_layer(out_dim, norm=norm, dim=2)
        self.act_i = get_activation_layer(act=act)

    def forward(self, x_r, x_i):
        r2r = self.conv_r(x_r)
        r2i = self.conv_i(x_r)
        i2i = self.conv_i(x_i)
        i2r = self.conv_r(x_i)

        real_out = r2r - i2i
        imag_out = i2r + r2i

        real_out = self.act_r(self.norm_r(real_out))
        imag_out = self.act_i(self.norm_i(imag_out))

        return real_out, imag_out

### ED_FNN

In [10]:
class ED_FNN_EnhancementStrategy(nn.Module):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__()
        self.win_len = win_len
        self.hop_len = hop_len
        self.fft_len = fft_len
        self.fft_half_len = fft_len // 2
        self.norm = norm
        self.act = act
        self.hidd_dim = hidd_dim


class ED_FNN_MagMapping(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    fcLayer(self.fft_half_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], self.fft_half_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')

    def forward(self, mag):
        hx = mag[:, 1:].permute(0, 2, 1)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        mag_out = functional.pad(out, [1, 0, 0, 0]).permute(0, 2, 1)
        return mag_out


class ED_FNN_MagMasking(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    fcLayer(self.fft_half_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], self.fft_half_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')

    def forward(self, mag):
        hx = mag[:, 1:].permute(0, 2, 1)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        mask_mag_out = functional.pad(out, [1, 0, 0, 0]).permute(0, 2, 1)
        mag_out = mag * mask_mag_out
        return mag_out


class ED_FNN_ComplexOperationMapping(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    complexFcLayer(self.fft_half_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    complexFcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    complexFcLayer(hidd_dim[idx-1], self.fft_half_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    complexFcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real = real_imag[0][:, 1:].permute(0, 2, 1)
        imag = real_imag[1][:, 1:].permute(0, 2, 1)

        real_out, imag_out = real, imag
        for idx, layer in enumerate(self.encoder):
            real_out, imag_out = layer(real_out, imag_out)

        for idx, layer in enumerate(self.decoder):
            real_out, imag_out = layer(real_out, imag_out)

        real_out = functional.pad(real_out, [1, 0, 0, 0]).permute(0, 2, 1)
        imag_out = functional.pad(imag_out, [1, 0, 0, 0]).permute(0, 2, 1)
        return real_out, imag_out


class ED_FNN_ComplexOperationMasking(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    complexFcLayer(self.fft_half_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    complexFcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    complexFcLayer(hidd_dim[idx-1], self.fft_half_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    complexFcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real, imag = real_imag[0], real_imag[1]
        hx_real = real[:, 1:].permute(0, 2, 1)
        hx_imag = imag[:, 1:].permute(0, 2, 1)

        real_mask_out, imag_mask_out = hx_real, hx_imag
        for idx, layer in enumerate(self.encoder):
            real_mask_out, imag_mask_out = layer(real_mask_out, imag_mask_out)

        for idx, layer in enumerate(self.decoder):
            real_mask_out, imag_mask_out = layer(real_mask_out, imag_mask_out)

        real_mask_out = functional.pad(real_mask_out, [1, 0, 0, 0]).permute(0, 2, 1)
        imag_mask_out = functional.pad(imag_mask_out, [1, 0, 0, 0]).permute(0, 2, 1)

        real_out = real * real_mask_out
        imag_out = imag * imag_mask_out
        return real_out, imag_out


class ED_FNN_ComplexChannelMapping(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    fcLayer(self.fft_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], self.fft_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )
        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real = real_imag[0][:, 1:].permute(0, 2, 1)
        imag = real_imag[1][:, 1:].permute(0, 2, 1)

        hx = torch.cat([real, imag], dim=2)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        real_out = out[..., :self.fft_half_len]
        imag_out = out[..., self.fft_half_len:]

        real_out = functional.pad(real_out, [1, 0, 0, 0]).permute(0, 2, 1)
        imag_out = functional.pad(imag_out, [1, 0, 0, 0]).permute(0, 2, 1)
        return real_out, imag_out


class ED_FNN_ComplexChannelMasking(ED_FNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, hidd_dim, norm, act):
        super().__init__(win_len, hop_len, fft_len, hidd_dim, norm, act)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_dim)):
            if idx == 0:
                self.encoder.append(
                    fcLayer(self.fft_len, hidd_dim[idx], norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx], norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_dim), 0, -1):
            if idx == 1:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], self.fft_len, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    fcLayer(hidd_dim[idx-1], hidd_dim[idx-2], norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real, imag = real_imag[0], real_imag[1]
        hx_real = real[:, 1:].permute(0, 2, 1)
        hx_imag = imag[:, 1:].permute(0, 2, 1)

        hx = torch.cat([hx_real, hx_imag], dim=2)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        real_mask_out = out[..., :self.fft_half_len]
        imag_mask_out = out[..., self.fft_half_len:]

        real_mask_out = functional.pad(real_mask_out, [1, 0, 0, 0]).permute(0, 2, 1)
        imag_mask_out = functional.pad(imag_mask_out, [1, 0, 0, 0]).permute(0, 2, 1)

        real_out = real * real_mask_out
        imag_out = imag * imag_mask_out
        return real_out, imag_out


class ED_FNN(nn.Module):
    def __init__(self, win_len=400, hop_len=100, fft_len=512, hidd_dim=[256, 512, 768, 768], processing_type='mag_mapping', norm='bn', act='relu'):
        super().__init__()

        self.win_len = win_len
        self.hop_len = hop_len
        self.fft_len = fft_len
        self.fft_half_len = fft_len // 2
        self.norm = norm
        self.act = act

        # Initialize different strategies
        self.strategies = {
            'mag_mapping': ED_FNN_MagMapping(win_len, hop_len, fft_len, hidd_dim, norm, act),
            'mag_masking': ED_FNN_MagMasking(win_len, hop_len, fft_len, hidd_dim, norm, act),
            'complex_operation_mapping': ED_FNN_ComplexOperationMapping(win_len, hop_len, fft_len, hidd_dim, norm, act),
            'complex_operation_masking': ED_FNN_ComplexOperationMasking(win_len, hop_len, fft_len, hidd_dim, norm, act),
            'complex_channel_mapping': ED_FNN_ComplexChannelMapping(win_len, hop_len, fft_len, hidd_dim, norm, act),
            'complex_channel_masking': ED_FNN_ComplexChannelMasking(win_len, hop_len, fft_len, hidd_dim, norm, act)
        }
        self.current_strategy = self.strategies[processing_type]
        self.stft = self.current_strategy.stft
        self.istft = self.current_strategy.istft

    def forward(self, x):
        return self.current_strategy(x)

### ED_CNN

In [11]:
class ED_CNN_EnhancementStrategy(nn.Module):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size=(3, 2), dilation=(1, 1)):
        super().__init__()
        self.win_len = win_len
        self.hop_len = hop_len
        self.fft_len = fft_len
        self.fft_half_len = fft_len // 2
        self.norm = norm
        self.act = act
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.hidd_ch = hidd_ch


class ED_CNN_MagMapping(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    convLayer(1, hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    convLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], 1, kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')

    def forward(self, mag):
        hx = mag.unqueeze(1)[:, :, 1:]

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        mag_out = functional.pad(out, [0, 0, 1, 0])
        return mag_out


class ED_CNN_MagMasking(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    convLayer(1, hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    convLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], 1, kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='real')

    def forward(self, mag):
        hx = mag.unsqueeze(1)[:, :, 1:]

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        mask_mag_out = functional.pad(out, [0, 0, 1, 0])
        mag_out = mag * mask_mag_out.squeeze(1)
        return mag_out


class ED_CNN_ComplexOperationMapping(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    complexConvLayer(1, hidd_ch[idx], kernel_size=kernel_size,
                                     dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    complexConvLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                                     dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    complexUpConvLayer(hidd_ch[idx - 1], 1, kernel_size=kernel_size,
                                       dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    complexUpConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                       dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real = real_imag[0].unsqueeze(1)[:, :, 1:]
        imag = real_imag[1].unsqueeze(1)[:, :, 1:]

        real_out, imag_out = real, imag
        for idx, layer in enumerate(self.encoder):
            real_out, imag_out = layer(real_out, imag_out)

        for idx, layer in enumerate(self.decoder):
            real_out, imag_out = layer(real_out, imag_out)

        real_out = functional.pad(real_out, [0, 0, 1, 0])
        imag_out = functional.pad(imag_out, [0, 0, 1, 0])
        return real_out.squeeze(1), imag_out.squeeze(1)


class ED_CNN_ComplexOperationMasking(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    complexConvLayer(1, hidd_ch[idx], kernel_size=kernel_size,
                                     dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    complexConvLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                                     dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    complexUpConvLayer(hidd_ch[idx - 1], 1, kernel_size=kernel_size,
                                       dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    complexUpConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                       dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real, imag = real_imag[0], real_imag[1]
        hx_real = real.unsqueeze(1)[:, :, 1:]
        hx_imag = imag.unsqueeze(1)[:, :, 1:]

        real_mask_out, imag_mask_out = hx_real, hx_imag
        for idx, layer in enumerate(self.encoder):
            real_mask_out, imag_mask_out = layer(real_mask_out, imag_mask_out)

        for idx, layer in enumerate(self.decoder):
            real_mask_out, imag_mask_out = layer(real_mask_out, imag_mask_out)

        real_mask_out = functional.pad(real_mask_out, [0, 0, 1, 0])
        imag_mask_out = functional.pad(imag_mask_out, [0, 0, 1, 0])

        real_out = real * real_mask_out.squeeze(1)
        imag_out = imag * imag_mask_out.squeeze(1)
        return real_out, imag_out


class ED_CNN_ComplexChannelMapping(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    convLayer(2, hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    convLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], 2, kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real = real_imag[0].unsqueeze(1)[:, :, 1:]
        imag = real_imag[1].unsqueeze(1)[:, :, 1:]

        hx = torch.cat([real, imag], dim=1)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        real_out = out[:, 0]
        imag_out = out[:, 1]

        real_out = functional.pad(real_out, [0, 0, 1, 0])
        imag_out = functional.pad(imag_out, [0, 0, 1, 0])
        return real_out, imag_out


class ED_CNN_ComplexChannelMasking(ED_CNN_EnhancementStrategy):
    def __init__(self, win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation):
        super().__init__(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation)
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()

        for idx in range(len(hidd_ch)):
            if idx == 0:
                self.encoder.append(
                    convLayer(2, hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.encoder.append(
                    convLayer(hidd_ch[idx - 1], hidd_ch[idx], kernel_size=kernel_size,
                              dilation=dilation, norm=self.norm, act=self.act)
                )

        for idx in range(len(hidd_ch), 0, -1):
            if idx == 1:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], 2, kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )
            else:
                self.decoder.append(
                    upConvLayer(hidd_ch[idx - 1], hidd_ch[idx - 2], kernel_size=kernel_size,
                                dilation=dilation, norm=self.norm, act=self.act)
                )

        # for feature extract
        self.stft = ConvSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')
        self.istft = ConviSTFT(self.win_len, self.hop_len, self.fft_len, feature_type='complex')

    def forward(self, real_imag):
        real, imag = real_imag[0], real_imag[1]
        hx_real = real.unsqueeze(1)[:, :, 1:]
        hx_imag = imag.unsqueeze(1)[:, :, 1:]

        hx = torch.cat([hx_real, hx_imag], dim=1)

        out = hx
        for idx, layer in enumerate(self.encoder):
            out = layer(out)

        for idx, layer in enumerate(self.decoder):
            out = layer(out)

        real_mask_out = out[:, 0]
        imag_mask_out = out[:, 1]

        real_mask_out = functional.pad(real_mask_out, [0, 0, 1, 0])
        imag_mask_out = functional.pad(imag_mask_out, [0, 0, 1, 0])

        real_out = real * real_mask_out
        imag_out = imag * imag_mask_out
        return real_out, imag_out


class ED_CNN(nn.Module):
    def __init__(self, win_len=400, hop_len=100, fft_len=512, processing_type='mag_mapping', norm='bn', act='relu',
                 hidd_ch=[32, 64, 128, 256], kernel_size=(3, 2), dilation=(1, 1)):
        super().__init__()

        # Initialize different strategies
        self.strategies = {
            'mag_mapping': ED_CNN_MagMapping(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation),
            'mag_masking': ED_CNN_MagMasking(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size, dilation),
            'complex_operation_mapping': ED_CNN_ComplexOperationMapping(win_len, hop_len, fft_len, norm, act, hidd_ch,
                                                                 kernel_size, dilation),
            'complex_operation_masking': ED_CNN_ComplexOperationMasking(win_len, hop_len, fft_len, norm, act, hidd_ch,
                                                                 kernel_size, dilation),
            'complex_channel_mapping': ED_CNN_ComplexChannelMapping(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size,
                                                             dilation),
            'complex_channel_masking': ED_CNN_ComplexChannelMasking(win_len, hop_len, fft_len, norm, act, hidd_ch, kernel_size,
                                                             dilation)
        }
        self.current_strategy = self.strategies[processing_type]
        self.stft = self.current_strategy.stft
        self.istft = self.current_strategy.istft

    def forward(self, x):
        return self.current_strategy(x)


## utils

### data load

In [12]:
def scan_directory(dir_name):
    if os.path.isdir(dir_name) is False:
        print("[Error] There is no directory '%s'." % dir_name)
        exit()

    addrs = []
    for subdir, dirs, files in os.walk(dir_name):
        for file in files:
            if file.endswith(".wav"):
                filepath = subdir + file
                addrs.append(filepath)
    return addrs

In [13]:
def mkdir(path):
    if not os.path.exists(path):
        os.makedirs(path)
    else:
        print("Already exist...")

In [14]:
def find_pair(noisy_file_name):
    clean_dirs = []
    for i in range(len(noisy_file_name)):
        addrs = noisy_file_name[i]
        if addrs.endswith(".wav"):
            clean_addrs = str(addrs).replace('noisy', 'clean')
            clean_dirs.append(clean_addrs)
    return clean_dirs

In [15]:
def addr2wav(noisy_addr, clean_addr, fs_set=16000, norm_type='None'):
    noisy_wav, fs = soundfile.read(noisy_addr)
    clean_wav, _ = soundfile.read(clean_addr)
    assert fs == fs_set
    noisy_wav, clean_wav = normalize_wav(noisy_wav, clean_wav, norm_type)
    return noisy_wav, clean_wav

In [16]:
def normalize_wav(noisy_wav, clean_wav, norm_type='min_max'):
    if norm_type == 'None':
        noisy_norm, clean_norm = noisy_wav, clean_wav
    elif norm_type == 'MinMax':
        min_val = min(np.min(clean_wav), np.min(noisy_wav))
        max_val = max(np.max(clean_wav), np.max(noisy_wav))
        clean_norm = (clean_wav - min_val) / (max_val - min_val)
        noisy_norm = (noisy_wav - min_val) / (max_val - min_val)
    elif norm_type == 'Zscore':
        mean_val = np.mean(np.concatenate((clean_wav, noisy_wav)))
        std_val = np.std(np.concatenate((clean_wav, noisy_wav)))
        clean_norm = (clean_wav - mean_val) / std_val
        noisy_norm = (noisy_wav - mean_val) / std_val
    elif norm_type == 'MaxAbs':
        max_abs_val = max(np.max(np.abs(clean_wav)), np.max(np.abs(noisy_wav)))
        clean_norm = clean_wav / max_abs_val
        noisy_norm = noisy_wav / max_abs_val
    elif norm_type == 'Robust':
        median_val = np.median(np.concatenate((clean_wav, noisy_wav)))
        iqr = np.percentile(np.concatenate((clean_wav, noisy_wav)), 75) - np.percentile(np.concatenate((clean_wav, noisy_wav)), 25)
        clean_norm = (clean_wav - median_val) / iqr
        noisy_norm = (noisy_wav - median_val) / iqr
    else:
        raise ValueError("Unknown normalization method specified")

    return noisy_norm, clean_norm

### model init

In [17]:
def get_arch(opt):
    arch = opt.arch
    type = opt.input_type + '_' + opt.target_type

    print('You choose ' + arch + '...')
    if arch == 'ED_FNN':
        model = ED_FNN(win_len=opt.win_len, hop_len=opt.hop_len, fft_len=opt.fft_len,
                       processing_type=type, norm=opt.norm_layer, act=opt.act)
    elif arch == 'ED_CNN':
        model = ED_CNN(win_len=opt.win_len, hop_len=opt.hop_len, fft_len=opt.fft_len,
                       processing_type=type, norm=opt.norm_layer, act=opt.act, hidd_ch=opt.hidd_ch,
                       kernel_size=opt.kernel_size, dilation=opt.dilation)

    else:
        raise Exception("Arch error!")

    return model

In [18]:
def get_train_mode(opt):
    input_type = opt.input_type
    loss_type = opt.loss_type

    print('You choose ' + loss_type + '...')
    if loss_type == 'base':  # single loss function
        if input_type == 'mag':
            trainer = base_mag_train
            validator = base_mag_valid
        elif (input_type == 'complex_operation') or (input_type == 'complex_channel'):
            trainer = base_complex_train
            validator = base_complex_valid
        else:
            raise Exception("Input type error! Please check the option")
    elif loss_type == 'joint':  # multiple(joint) loss function
        if input_type == 'mag':
            trainer = joint_mag_train
            validator = joint_mag_valid
        elif (input_type == 'complex_operation') or (input_type == 'complex_channel'):
            trainer = joint_complex_train
            validator = joint_complex_valid
        else:
            raise Exception("Input type error! Please check the option")
    else:
        raise Exception("Loss type error!")

    return trainer, validator

In [19]:
def get_loss(opt):
    from torch.nn import L1Loss
    from torch.nn.functional import mse_loss
    loss_oper = opt.loss_oper

    print('You choose loss operation with ' + loss_oper + '...')
    if loss_oper == 'l1':
        loss_calculator = L1Loss()
    elif loss_oper == 'l2':
        loss_calculator = mse_loss
    else:
        raise Exception("Arch error!")

    return loss_calculator

In [20]:
def get_normalization_layer(ch, dim=1, norm='bn'):
    if norm == 'bn':  # batch norm
        if dim == 1:
            from torch.nn import BatchNorm1d
            return BatchNorm1d(ch)
        elif dim == 2:
            from torch.nn import BatchNorm2d
            return BatchNorm2d(ch)
        else:
            raise Exception("Dimension error! Please check the option")
    elif norm == 'in':  # instance norm
        from torch.nn import GroupNorm
        return GroupNorm(ch, ch)
    elif norm == 'gn': # group norm
        from torch.nn import GroupNorm
        groups = 2
        return GroupNorm(groups, ch)
    elif norm == 'ln': # layer norm
        from torch.nn import GroupNorm
        return GroupNorm(1, ch)
    else:
        raise Exception("Normalization error! Please check the option")

In [21]:
def get_activation_layer(act='relu'):
    if act == 'sigmoid':
        from torch.nn import Sigmoid
        return Sigmoid()
    elif act == 'relu':
        from torch.nn import ReLU
        return ReLU()
    elif act == 'leakyrelu':
        from torch.nn import LeakyReLU
        return LeakyReLU()
    elif act == 'tanh':
        from torch.nn import Tanh
        return Tanh()
    elif act == 'prelu':
        from torch.nn import PReLU
        return PReLU()
    else:
        raise Exception("Activation error! Please check the option")

### trainer

In [22]:
def base_mag_train(model, train_loader, loss_calculator, optimizer, writer, EPOCH, DEVICE, opt):
    # initialization
    train_loss = 0
    batch_num = 0

    # train
    model.train()

    for inputs, targets in Bar(train_loader):
        batch_num += 1

        # to cuda
        inputs = inputs.float().to(DEVICE)
        targets = targets.float().to(DEVICE)

        # generator
        input_mags, input_phase = model.stft(inputs)
        input_mags = power_compress_mag(input_mags, compression_factor=opt.cmpr)

        out_mags = model(input_mags)

        clean_mags, clean_phase = model.stft(targets)
        clean_mags = power_compress_mag(clean_mags)

        loss = loss_calculator(out_mags, clean_mags)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    train_loss /= batch_num

    # tensorboard
    writer.log_train_loss('mag', train_loss / batch_num, EPOCH)

    return train_loss

In [23]:
def base_complex_train(model, train_loader, loss_calculator, optimizer, writer,
                   EPOCH, DEVICE, opt):
    # initialization
    train_loss = 0
    batch_num = 0

    # train
    model.train()

    for inputs, targets in Bar(train_loader):
        batch_num += 1

        # to cuda
        inputs = inputs.float().to(DEVICE)
        targets = targets.float().to(DEVICE)

        # generator
        input_specs = model.stft(inputs)
        input_real, input_imag = power_compress(input_specs, cut_len=opt.fft_len // 2 + 1, compression_factor=opt.cmpr)

        out_real, out_imag = model((input_real, input_imag))
        out_mags = torch.sqrt(out_real ** 2 + out_imag ** 2)

        clean_specs = model.stft(targets)
        clean_real, clean_imag = power_compress(clean_specs, cut_len=opt.fft_len // 2 + 1, compression_factor=opt.cmpr)
        clean_mag = torch.sqrt(clean_real ** 2 + clean_imag ** 2 + 1e-7)

        loss = loss_calculator(out_mags, clean_mag)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    train_loss /= batch_num

    # tensorboard
    writer.log_train_loss('mag', train_loss / batch_num, EPOCH)

    return train_loss

In [24]:
def joint_mag_train(model, train_loader, loss_calculator, optimizer, writer, EPOCH, DEVICE, opt):
    # initialization
    train_loss = 0
    train_mag_loss = 0
    train_real_imag_loss = 0
    train_real_loss = 0
    train_imag_loss = 0
    batch_num = 0

    # train
    model.train()

    for inputs, targets in Bar(train_loader):
        batch_num += 1

        # to cuda
        inputs = inputs.float().to(DEVICE)
        targets = targets.float().to(DEVICE)

        # generator
        input_mags, input_phase = model.stft(inputs)
        input_mags = power_compress_mag(input_mags, compression_factor=opt.cmpr)

        out_mags = model(input_mags)
        out_real = out_mags * torch.cos(input_phase)
        out_imag = out_mags * torch.sin(input_phase)

        clean_mags, clean_phase = model.stft(targets)
        clean_mags = power_compress_mag(clean_mags, compression_factor=opt.cmpr)
        clean_real = clean_mags * torch.cos(clean_phase)
        clean_imag = clean_mags * torch.sin(clean_phase)

        mag_loss = loss_calculator(out_mags, clean_mags)

        real_loss = loss_calculator(out_real, clean_real)
        imag_loss = loss_calculator(out_imag, clean_imag)
        real_imag_loss = real_loss + imag_loss

        loss = opt.c[0] * real_imag_loss + opt.c[1] * mag_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_mag_loss += mag_loss.item()
        train_real_imag_loss += real_imag_loss.item()
        train_real_loss += real_loss.item()
        train_imag_loss += imag_loss.item()
    train_loss /= batch_num

    # tensorboard
    writer.log_train_loss('mag+real+imag', train_loss, EPOCH)
    writer.log_train_loss('mag', train_mag_loss / batch_num, EPOCH)
    writer.log_train_loss('real+imag', train_real_imag_loss / batch_num, EPOCH)
    writer.log_train_loss('real', train_real_loss / batch_num, EPOCH)
    writer.log_train_loss('imag', train_imag_loss / batch_num, EPOCH)

    return train_loss

In [25]:
def joint_complex_train(model, train_loader, loss_calculator, optimizer, writer,
                             EPOCH, DEVICE, opt):
    # initialization
    train_loss = 0
    train_mag_loss = 0
    train_real_imag_loss = 0
    train_real_loss = 0
    train_imag_loss = 0
    batch_num = 0

    # train
    model.train()

    for inputs, targets in Bar(train_loader):
        batch_num += 1

        # to cuda
        inputs = inputs.float().to(DEVICE)
        targets = targets.float().to(DEVICE)

        # generator
        input_specs = model.stft(inputs)
        input_real, input_imag = power_compress(input_specs, cut_len=opt.fft_len // 2 + 1, compression_factor=opt.cmpr)

        out_real, out_imag = model((input_real, input_imag))

        clean_specs = model.stft(targets)
        clean_real, clean_imag = power_compress(clean_specs, cut_len=opt.fft_len // 2 + 1, compression_factor=opt.cmpr)
        clean_mag = torch.sqrt(clean_real ** 2 + clean_imag ** 2 + 1e-7)

        out_mags = torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-7)

        mag_loss = loss_calculator(out_mags, clean_mag)

        real_loss = loss_calculator(out_real, clean_real)
        imag_loss = loss_calculator(out_imag, clean_imag)
        real_imag_loss = real_loss + imag_loss

        loss = opt.c[0] * real_imag_loss + opt.c[1] * mag_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_mag_loss += mag_loss.item()
        train_real_imag_loss += real_imag_loss.item()
        train_real_loss += real_loss.item()
        train_imag_loss += imag_loss.item()
    train_loss /= batch_num

    # tensorboard
    writer.log_train_loss('mag+real+imag', train_loss, EPOCH)
    writer.log_train_loss('mag', train_mag_loss / batch_num, EPOCH)
    writer.log_train_loss('real+imag', train_real_imag_loss / batch_num, EPOCH)
    writer.log_train_loss('real', train_real_loss / batch_num, EPOCH)
    writer.log_train_loss('imag', train_imag_loss / batch_num, EPOCH)

    return train_loss

In [26]:
def base_mag_valid(model, valid_loader, loss_calculator, writer, EPOCH, DEVICE, opt):
    # initialization
    valid_loss = 0
    batch_num = 0
    avg_pesq = 0
    avg_stoi = 0

    # validation
    model.eval()

    with torch.no_grad():
        for inputs, targets in Bar(valid_loader):
            batch_num += 1

            # to cuda
            inputs = inputs.float().to(DEVICE)
            targets = targets.float().to(DEVICE)

            # generator
            input_mags, input_phase = model.stft(inputs)
            input_mags = power_compress_mag(input_mags, compression_factor=opt.cmpr)

            out_mags = model(input_mags)

            out_mags_uncompressed = power_uncompress_mag(out_mags, compression_factor=opt.cmpr)
            out_real = out_mags_uncompressed * torch.cos(input_phase)
            out_imag = out_mags_uncompressed * torch.sin(input_phase)
            out_specs = torch.cat([out_real, out_imag], dim=1)
            outputs = model.istft(out_specs)
            outputs = outputs.squeeze(1)

            clean_mags, clean_phase = model.stft(targets)
            clean_mags = power_compress_mag(clean_mags)

            loss = loss_calculator(out_mags, clean_mags)

            clean_wavs = targets.cpu().detach().numpy()[:, :outputs.size(1)]
            enhanced_wavs = outputs.cpu().detach().numpy()

            valid_loss += loss

            # get score
            pesq = cal_pesq_batch(clean_wavs, enhanced_wavs)
            stoi = cal_stoi_batch(clean_wavs, enhanced_wavs)

            avg_pesq += pesq
            avg_stoi += stoi

        valid_loss /= batch_num
        avg_pesq /= batch_num
        avg_stoi /= batch_num

    # tensorboard
    writer.log_valid_loss('mag', valid_loss / batch_num, EPOCH)
    writer.log_score('PESQ', avg_pesq, EPOCH)
    writer.log_score('STOI', avg_stoi, EPOCH)
    writer.log_wav(inputs[0], targets[0], outputs[0], EPOCH)
    writer.log_spectrogram(inputs[0], targets[0], outputs[0], EPOCH)

    return valid_loss, avg_pesq, avg_stoi

In [27]:
def base_complex_valid(model, valid_loader, loss_calculator, writer, EPOCH, DEVICE, opt):
    # initialization
    valid_loss = 0
    batch_num = 0
    avg_pesq = 0
    avg_stoi = 0

    # validation
    model.eval()

    with torch.no_grad():
        for inputs, targets in Bar(valid_loader):
            batch_num += 1

            # to cuda
            inputs = inputs.float().to(DEVICE)
            targets = targets.float().to(DEVICE)

            # generator
            input_specs = model.stft(inputs)
            input_real, input_imag = power_compress(input_specs, cut_len=opt.fft_len // 2 + 1,
                                                    compression_factor=opt.cmpr)

            out_real, out_imag = model((input_real, input_imag))
            out_mags = torch.sqrt(out_real ** 2 + out_imag ** 2)
            out_specs = power_uncompress(out_real, out_imag, compression_factor=opt.cmpr)
            outputs = model.istft(out_specs)
            outputs = outputs.squeeze(1)

            clean_specs = model.stft(targets)
            clean_real, clean_imag = power_compress(clean_specs, compression_factor=opt.cmpr)
            clean_mag = torch.sqrt(clean_real ** 2 + clean_imag ** 2 + 1e-7)

            loss = loss_calculator(out_mags, clean_mag)

            clean_wavs = targets.cpu().detach().numpy()[:, :outputs.size(1)]
            enhanced_wavs = outputs.cpu().detach().numpy()

            valid_loss += loss

            # get score
            pesq = cal_pesq_batch(clean_wavs, enhanced_wavs)
            stoi = cal_stoi_batch(clean_wavs, enhanced_wavs)

            avg_pesq += pesq
            avg_stoi += stoi

        valid_loss /= batch_num
        avg_pesq /= batch_num
        avg_stoi /= batch_num

    # tensorboard
    writer.log_valid_loss('mag', valid_loss / batch_num, EPOCH)
    writer.log_score('PESQ', avg_pesq, EPOCH)
    writer.log_score('STOI', avg_stoi, EPOCH)
    writer.log_wav(inputs[0], targets[0], outputs[0], EPOCH)
    writer.log_spectrogram(inputs[0], targets[0], outputs[0], EPOCH)

    return valid_loss, avg_pesq, avg_stoi

In [28]:
def joint_mag_valid(model, valid_loader, loss_calculator, writer, EPOCH, DEVICE, opt):
    # initialization
    valid_loss = 0
    valid_mag_loss = 0
    valid_real_imag_loss = 0
    valid_real_loss = 0
    valid_imag_loss = 0
    batch_num = 0
    avg_pesq = 0
    avg_stoi = 0

    # validation
    model.eval()

    with torch.no_grad():
        for inputs, targets in Bar(valid_loader):
            batch_num += 1

            # to cuda
            inputs = inputs.float().to(DEVICE)
            targets = targets.float().to(DEVICE)

            # generator
            input_mags, input_phase = model.stft(inputs)
            input_mags = power_compress_mag(input_mags, compression_factor=opt.cmpr)

            out_mags = model(input_mags)
            out_real = out_mags * torch.cos(input_phase)
            out_imag = out_mags * torch.sin(input_phase)

            clean_mags, clean_phase = model.stft(targets)
            clean_mags = power_compress_mag(clean_mags, compression_factor=opt.cmpr)
            clean_real = clean_mags * torch.cos(clean_phase)
            clean_imag = clean_mags * torch.sin(clean_phase)

            mag_loss = loss_calculator(out_mags, clean_mags)

            real_loss = loss_calculator(out_real, clean_real)
            imag_loss = loss_calculator(out_imag, clean_imag)
            real_imag_loss = real_loss + imag_loss

            loss = opt.c[0] * real_imag_loss + opt.c[1] * mag_loss

            out_specs = power_uncompress(out_real, out_imag)
            outputs = model.istft(out_specs)
            outputs = outputs.squeeze(1)
            clean_wavs = targets.cpu().detach().numpy()[:, :outputs.size(1)]
            enhanced_wavs = outputs.cpu().detach().numpy()

            valid_loss += loss
            valid_mag_loss += mag_loss
            valid_real_imag_loss += real_imag_loss
            valid_real_loss += real_loss
            valid_imag_loss += imag_loss

            # get score
            pesq = cal_pesq_batch(clean_wavs, enhanced_wavs)
            stoi = cal_stoi_batch(clean_wavs, enhanced_wavs)

            avg_pesq += pesq
            avg_stoi += stoi

        valid_loss /= batch_num
        avg_pesq /= batch_num
        avg_stoi /= batch_num

    # tensorboard
    writer.log_valid_loss('mag+real+imag', valid_loss, EPOCH)
    writer.log_valid_loss('mag', valid_mag_loss / batch_num, EPOCH)
    writer.log_valid_loss('real+imag', valid_real_imag_loss / batch_num, EPOCH)
    writer.log_valid_loss('real', valid_real_loss / batch_num, EPOCH)
    writer.log_valid_loss('imag', valid_imag_loss / batch_num, EPOCH)
    writer.log_score('PESQ', avg_pesq, EPOCH)
    writer.log_score('STOI', avg_stoi, EPOCH)
    writer.log_wav(inputs[0], targets[0], outputs[0], EPOCH)
    writer.log_spectrogram(inputs[0], targets[0], outputs[0], EPOCH)

    return valid_loss, avg_pesq, avg_stoi

In [29]:
def joint_complex_valid(model, valid_loader, loss_calculator, writer, EPOCH, DEVICE, opt):
    # initialization
    valid_loss = 0
    valid_mag_loss = 0
    valid_real_imag_loss = 0
    valid_real_loss = 0
    valid_imag_loss = 0
    batch_num = 0
    avg_pesq = 0
    avg_stoi = 0

    # validation
    model.eval()

    with torch.no_grad():
        for inputs, targets in Bar(valid_loader):
            batch_num += 1

            # to cuda
            inputs = inputs.float().to(DEVICE)
            targets = targets.float().to(DEVICE)

            # generator
            input_specs = model.stft(inputs)
            input_real, input_imag = power_compress(input_specs, cut_len=opt.fft_len // 2 + 1,
                                                    compression_factor=opt.cmpr)

            out_real, out_imag = model((input_real, input_imag))
            out_mags = torch.sqrt(out_real ** 2 + out_imag ** 2 + 1e-7)
            out_specs = power_uncompress(out_real, out_imag, compression_factor=opt.cmpr)
            outputs = model.istft(out_specs)
            outputs = outputs.squeeze(1)

            clean_specs = model.stft(targets)
            clean_real, clean_imag = power_compress(clean_specs, cut_len=opt.fft_len // 2 + 1,
                                                    compression_factor=opt.cmpr)
            clean_mag = torch.sqrt(clean_real ** 2 + clean_imag ** 2 + 1e-7)

            mag_loss = loss_calculator(out_mags, clean_mag)

            real_loss = loss_calculator(out_real, clean_real)
            imag_loss = loss_calculator(out_imag, clean_imag)
            real_imag_loss = real_loss + imag_loss

            loss = opt.c[0] * real_imag_loss + opt.c[1] * mag_loss

            clean_wavs = targets.cpu().detach().numpy()[:, :outputs.size(1)]
            enhanced_wavs = outputs.cpu().detach().numpy()

            valid_loss += loss
            valid_mag_loss += mag_loss
            valid_real_imag_loss += real_imag_loss
            valid_real_loss += real_loss
            valid_imag_loss += imag_loss

            # get score
            pesq = cal_pesq_batch(clean_wavs, enhanced_wavs)
            stoi = cal_stoi_batch(clean_wavs, enhanced_wavs)

            avg_pesq += pesq
            avg_stoi += stoi

        valid_loss /= batch_num
        avg_pesq /= batch_num
        avg_stoi /= batch_num

    # tensorboard
    writer.log_valid_loss('mag+real+imag', valid_loss, EPOCH)
    writer.log_valid_loss('mag', valid_mag_loss / batch_num, EPOCH)
    writer.log_valid_loss('real+imag', valid_real_imag_loss / batch_num, EPOCH)
    writer.log_valid_loss('real', valid_real_loss / batch_num, EPOCH)
    writer.log_valid_loss('imag', valid_imag_loss / batch_num, EPOCH)
    writer.log_score('PESQ', avg_pesq, EPOCH)
    writer.log_score('STOI', avg_stoi, EPOCH)
    writer.log_wav(inputs[0], targets[0], outputs[0], EPOCH)
    writer.log_spectrogram(inputs[0], targets[0], outputs[0], EPOCH)

    return valid_loss, avg_pesq, avg_stoi

###scores

In [30]:
def cal_pesq(clean_wav, dirty_wav, FS=16000):
    try:
        pesq_score = pesq(FS, clean_wav, dirty_wav, "wb")
    except:
        print(' No utterances error')
        pesq_score = -1
    return pesq_score


def cal_pesq_batch(clean_wavs, dirty_wavs, FS=16000):
    pesq_score = Parallel(n_jobs=1)(delayed(cal_pesq)(c, n, FS=FS) for c, n in zip(clean_wavs, dirty_wavs))
    pesq_score = np.array(pesq_score)
    return np.mean(pesq_score)


def cal_stoi_batch(clean_wavs, dirty_wavs, FS=16000):
    stoi_score = Parallel(n_jobs=1)(delayed(stoi)(c, n, FS, extended=False) for c, n in zip(clean_wavs, dirty_wavs))
    stoi_score = np.array(stoi_score)
    return np.mean(stoi_score)

###stft

In [31]:
# this is from conv_stft https://github.com/huyanxin/DeepComplexCRN


def init_kernels(win_len, fft_len, win_type=None, invers=False):
    if win_type == 'None' or win_type is None:
        window = np.ones(win_len)
    else:
        window = get_window(win_type, win_len, fftbins=True)  # **0.5

    N = fft_len
    fourier_basis = np.fft.rfft(np.eye(N))[:win_len]
    real_kernel = np.real(fourier_basis)
    imag_kernel = np.imag(fourier_basis)
    kernel = np.concatenate([real_kernel, imag_kernel], 1).T

    if invers:
        kernel = np.linalg.pinv(kernel).T

    kernel = kernel * window
    kernel = kernel[:, None, :]
    return torch.from_numpy(kernel.astype(np.float32)), torch.from_numpy(window[None, :, None].astype(np.float32))


class ConvSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real'):
        super(ConvSTFT, self).__init__()

        if fft_len is None:
            self.fft_len = np.int(2 ** np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len

        kernel, _ = init_kernels(win_len, self.fft_len, win_type)
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.stride = win_inc
        self.win_len = win_len
        self.dim = self.fft_len

    def forward(self, inputs):
        if inputs.dim() == 2:
            inputs = torch.unsqueeze(inputs, 1)
        inputs = functional.pad(inputs, [self.win_len - self.stride, self.win_len - self.stride])
        outputs = functional.conv1d(inputs, self.weight, stride=self.stride)

        if self.feature_type == 'complex':
            return outputs
        else:
            dim = self.dim // 2 + 1
            real = outputs[:, :dim, :]
            imag = outputs[:, dim:, :]
            mags = torch.sqrt(real ** 2 + imag ** 2)
            phase = torch.atan2(imag, real)
            return mags, phase  # , real, imag


class ConviSTFT(nn.Module):

    def __init__(self, win_len, win_inc, fft_len=None, win_type='hamming', feature_type='real'):
        super(ConviSTFT, self).__init__()
        if fft_len is None:
            self.fft_len = np.int(2 ** np.ceil(np.log2(win_len)))
        else:
            self.fft_len = fft_len
        kernel, window = init_kernels(win_len, self.fft_len, win_type, invers=True)
        self.register_buffer('weight', kernel)
        self.feature_type = feature_type
        self.win_type = win_type
        self.win_len = win_len
        self.stride = win_inc
        self.dim = self.fft_len
        self.register_buffer('window', window)
        self.register_buffer('enframe', torch.eye(win_len)[:, None, :])

    def forward(self, inputs, phase=None):
        """
        inputs : [B, N+2, T] (complex spec) or [B, N//2+1, T] (mags)
        phase: [B, N//2+1, T] (if not none)
        """

        if phase is not None:
            real = inputs * torch.cos(phase)
            imag = inputs * torch.sin(phase)
            inputs = torch.cat([real, imag], 1)

        outputs = functional.conv_transpose1d(inputs, self.weight, stride=self.stride)

        # this is from torch-stft: https://github.com/pseeth/torch-stft
        t = self.window.repeat(1, 1, inputs.size(-1)) ** 2
        coff = functional.conv_transpose1d(t, self.enframe, stride=self.stride)

        outputs = outputs / (coff + 1e-8)

        outputs = outputs[..., self.win_len - self.stride:-(self.win_len - self.stride)]

        return outputs

###tensorboard

In [32]:
class Writer(SummaryWriter):
    def __init__(self, logdir):
        super(Writer, self).__init__(logdir)

    def log_train_loss(self, loss_type, train_loss, step):
        self.add_scalar('train_{}_loss'.format(loss_type), train_loss, step)

    def log_valid_loss(self, loss_type, valid_loss, step):
        self.add_scalar('valid_{}_loss'.format(loss_type), valid_loss, step)

    def log_score(self, metrics_name, metrics, step):
        self.add_scalar(metrics_name, metrics, step)

    def log_wav(self, noisy_wav, clean_wav, enhanced_wav, step):
        # <Audio>
        self.add_audio('noisy_wav', noisy_wav, step, sample_rate=16000)
        self.add_audio('clean_target_wav', clean_wav, step, sample_rate=16000)
        self.add_audio('enhanced_wav', enhanced_wav, step, sample_rate=16000)

    def log_spectrogram(self, noisy_wav, clean_wav, enhanced_wav, step):
        # <Audio>
        self.plot_spectrogram('noisy_wav_spectrogram', noisy_wav, step, sample_rate=16000)
        self.plot_spectrogram('clean_target_wav_spectrogram', clean_wav, step, sample_rate=16000)
        self.plot_spectrogram('enhanced_wav_spectrogram', enhanced_wav, step, sample_rate=16000)

    def plot_spectrogram(self, tag, wav, step, sample_rate=16000):
        wav = wav.cpu().numpy()

        # Convert wav to spectrogram
        D = librosa.stft(wav)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

        # Plot
        plt.figure(figsize=(10, 4))
        librosa.display.specshow(S_db, sr=sample_rate, x_axis='time', y_axis='log')
        plt.colorbar(format='%+2.0f dB')
        plt.title('Linear-frequency power spectrogram')
        plt.tight_layout()

        # Save to buffer
        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)

        # Log to TensorBoard
        self.add_image(tag, np.array(PIL.Image.open(buf)), step, dataformats='HWC')

        # Close plot
        plt.close()

###progress

In [33]:
def optimizer_to(optim, device):
    for param in optim.state.values():
        # Not sure there are any global tensors in the state dict
        if isinstance(param, torch.Tensor):
            param.data = param.data.to(device)
            if param._grad is not None:
                param._grad.data = param._grad.data.to(device)
        elif isinstance(param, dict):
            for subparam in param.values():
                if isinstance(subparam, torch.Tensor):
                    subparam.data = subparam.data.to(device)
                    if subparam._grad is not None:
                        subparam._grad.data = subparam._grad.data.to(device)

In [34]:
# calculate the size of total network
def cal_total_params(our_model):
    total_parameters = 0
    for variable in our_model.parameters():
        shape = variable.size()
        variable_parameters = 1
        for dim in shape:
            variable_parameters *= dim
        total_parameters += variable_parameters

    return total_parameters

In [35]:
class Bar(object):
    def __init__(self, dataloader):
        if not hasattr(dataloader, 'dataset'):
            raise ValueError('Attribute `dataset` not exists in dataloder.')
        if not hasattr(dataloader, 'batch_size'):
            raise ValueError('Attribute `batch_size` not exists in dataloder.')

        self.dataloader = dataloader
        self.iterator = iter(dataloader)
        self.dataset = dataloader.dataset
        self.batch_size = dataloader.batch_size
        self._idx = 0
        self._batch_idx = 0
        self._time = []
        self._DISPLAY_LENGTH = 50

    def __len__(self):
        return len(self.dataloader)

    def __iter__(self):
        return self

    def __next__(self):
        if len(self._time) < 2:
            self._time.append(time.time())

        self._batch_idx += self.batch_size
        if self._batch_idx > len(self.dataset):
            self._batch_idx = len(self.dataset)

        try:
            batch = next(self.iterator)
            self._display()
        except StopIteration:
            raise StopIteration()

        self._idx += 1
        if self._idx >= len(self.dataloader):
            self._reset()

        return batch

    def _display(self):
        if len(self._time) > 1:
            t = (self._time[-1] - self._time[-2])
            eta = t * (len(self.dataloader) - self._idx)
        else:
            eta = 0

        rate = self._idx / len(self.dataloader)
        len_bar = int(rate * self._DISPLAY_LENGTH)
        bar = ('=' * len_bar + '>').ljust(self._DISPLAY_LENGTH, '.')
        idx = str(self._batch_idx).rjust(len(str(len(self.dataset))), ' ')

        tmpl = '\r{}/{}: [{}] - ETA {:.1f}s'.format(
            idx,
            len(self.dataset),
            bar,
            eta
        )
        print(tmpl, end='')
        if self._batch_idx == len(self.dataset):
            print()

    def _reset(self):
        self._idx = 0
        self._batch_idx = 0
        self._time = []

In [36]:
def power_compress(x, cut_len=257, compression_factor=0.2):
    real = x[:, :cut_len]
    imag = x[:, cut_len:]
    mags = torch.sqrt(real ** 2 + imag ** 2 + 1e-7)
    phase = torch.atan2(imag, real)
    mags = mags ** compression_factor + 1e-7
    real_compress = mags * torch.cos(phase)
    imag_compress = mags * torch.sin(phase)
    return real_compress, imag_compress

def power_uncompress(real, imag, compression_factor=0.2):
    mags = torch.sqrt(real ** 2 + imag ** 2 + 1e-7)
    phase = torch.atan2(imag, real)
    mags = mags ** (1. / compression_factor) + 1e-7
    real_compress = mags * torch.cos(phase)
    imag_compress = mags * torch.sin(phase)
    return torch.cat([real_compress, imag_compress], 1)

In [37]:
def power_compress_mag(mags, compression_factor=0.2):
    return mags ** compression_factor + 1e-7

def power_uncompress_mag(mags, compression_factor=0.2):
    return mags ** (1. / compression_factor) + 1e-7

## Train interface

In [ ]:
######################################################################################################################
#                                                  Parser init                                                       #
######################################################################################################################
print(args)

######################################################################################################################
#                                    Set a model (check point) and a log folder                                      #
######################################################################################################################
mkdir('./log')
date = datetime.datetime.now()
log_dir = os.path.join('./log', args.arch + '_' + str(date.month) + str(date.day) + '_' + args.env)
mkdir(log_dir)
print("Now time is : ", date.isoformat())
tboard_dir = os.path.join(log_dir, 'logs')
model_dir = os.path.join(log_dir, 'models')
mkdir(model_dir)  # make a dir if there is no dir (given path)
mkdir(tboard_dir)

######################################################################################################################
#                                                   Model init                                                       #
######################################################################################################################
# set device
DEVICE = torch.device(args.device)

# set seeds
random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)
torch.cuda.manual_seed_all(1234)

# define model
model = get_arch(args)

total_params = cal_total_params(model)
print('total params : %d (%.2f M, %.2f MBytes)\n' %
      (total_params,
       total_params / 1000000.0,
       total_params * 4.0 / 1000000.0))

# define loss type
trainer, validator = get_train_mode(args)
loss_calculator = get_loss(args)

# define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr_initial)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=args.decay_epoch, gamma=0.5)

# load the params if there is pretrained model
epoch_start_idx = 1
if args.pretrained:
    print('Load the pretrained model...')
    chkpt = torch.load(args.pretrain_model_path)
    model.load_state_dict(chkpt['model'])
    if not args.pretrained_init:
        optimizer.load_state_dict(chkpt['optimizer'])
        epoch_start_idx = chkpt['epoch'] + 1
        print('Resuming Start Epoch: ', epoch_start_idx)

        optimizer_to(optimizer, DEVICE)

model = model.to(DEVICE)
######################################################################################################################
#                                               Create Dataloader                                                    #
######################################################################################################################
train_loader = create_dataloader(args, mode='train')
valid_loader = create_dataloader(args, mode='valid')
print("Sizeof training set: ", train_loader.__len__(),
      ", sizeof validation set: ", valid_loader.__len__())

######################################################################################################################
######################################################################################################################
#                                             Main program - train                                                   #
######################################################################################################################
######################################################################################################################
writer = Writer(tboard_dir)
train_log_fp = open(model_dir + '/train_log.txt', 'a')

print('Train start...')
for epoch in range(epoch_start_idx, args.nepoch + 1):
    st_time = time.time()

    # train
    train_loss = trainer(model, train_loader, loss_calculator, optimizer,
                         writer, epoch, DEVICE, args)

    # save checkpoint file to resume training
    save_path = str(model_dir + '/chkpt_%d.pt' % epoch)
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch
    }, save_path)

    # validate
    valid_loss, pesq, stoi = validator(model, valid_loader, loss_calculator,
                                       writer, epoch, DEVICE, args)

    print('EPOCH[{}] T {:.6f} |  V {:.6f}  takes {:.3f} seconds'
          .format(epoch, train_loss, valid_loss, time.time() - st_time))
    print('PESQ {:.6f} | STOI {:.6f}'.format(pesq, stoi))

    # write train log
    train_log_fp.write('EPOCH[{}] T {:.6f} |  V {:.6f}  takes {:.3f} seconds'
                       .format(epoch, train_loss, valid_loss, time.time() - st_time))
    train_log_fp.write('PESQ {:.6f} | STOI {:.6f}\n'.format(pesq, stoi))

    # scheduler
    scheduler.step()

print('Training has been finished.')
train_log_fp.close()